# 19 · Snap sites to the network against pyproj

**Question.** Does Fieldwork's `snap_to_network` widget pick the nearest junction, and report
honestly how far it moved the site?

The first step of the isochrone chain. A snap is a nearest-neighbour search and a distance,
both exact — the reference projects the 8 pumps and the 738 network nodes with pyproj and takes
the smallest Euclidean distance.

Criteria fixed before running: the same node for every site, the distance moved within 5 mm,
the returned position being the *node's* and not the site's, the counts exact, and no site
beyond the 150 m limit.

**5 mm, not 1 mm, and the reason matters.** The widget reports `snapDistanceM` as
`Number(d.toFixed(2))` — it computes in full precision and *publishes* centimetres. 5 mm is
half of its last reported digit, not a statement about its accuracy. The first version of this
check asked for 1 mm and failed at 3.7 mm, which was rounding and nothing else.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_19_snap import run

report = run(Path.cwd() / "fixtures" / "snap-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["sites"], "sites against", report["networkNodes"], "nodes")
print()
print(f"{'site':<22} {'node':<10} {'moved m':>9} {'margin m':>10}")
for row in report["perSite"]:
    print(f"{row['name'][:22]:<22} {row['fieldworkNode']:<10} "
          f"{row['fieldworkDistanceM']:>9.2f} {row['marginToSecondNearestM']:>10.2f}")
print()
print(f"worst distance error {report['worstDistanceErrorMetres']:.3e} m "
      f"(reported precision {report['reportedPrecisionM']} m)")
print(f"equidistant ties {report['equidistantTies']}, "
      f"smallest margin to the second nearest {report['smallestMarginToSecondNearestM']:.2f} m")

## The margin column is the interesting one

A snap is a choice between nodes, and the check reports how much closer the chosen node is than
the runner-up. **One site's margin is 0.43 m.** No tie, so the answer is not ambiguous — but a
margin that small means the choice is decided by less than the width of a pavement, and a
different projection, a different datum, or a slightly different node position would flip it.

That is reported rather than hidden because a check saying only "the nodes agree" would imply a
robustness this data does not have.

## Dive deeper · snapping moves the site

Everything downstream starts from the junction, not from where the pump stood. The question is
whether that displacement matters against the distances the chain then measures.

In [ ]:
fixture = json.loads((Path.cwd() / "fixtures" / "snap-soho-001.json").read_text())
budget = 5 * 75   # the five-minute band: 5 minutes at 75 m per minute

moved = [row["fieldworkDistanceM"] for row in report["perSite"]]
print(f"sites snapped                   {len(moved)}")
print(f"distance moved, least to most   {min(moved):.2f} to {max(moved):.2f} m")
print(f"five-minute band budget         {budget} m")
print()
print(f"The worst displacement is {max(moved) / budget:.1%} of the five-minute budget, so a pump")
print(f"modelled at its junction starts up to {max(moved):.1f} m from where it stood -- about")
print(f"{max(moved) / 75 * 60:.0f} seconds of walking at the assumed speed.")
print()
print("Small, and not nothing: it is the same order as the 25 m corridor half-width the")
print("isochrone chain uses, which is itself a declared claim rather than a measurement.")

### What the displacement costs, and who pays it

**Added 2026-10-14.** The section above measures how far each site moves and compares it to the
five-minute budget. The question it leaves open — *whether that displacement matters* — has an
answer, and it is not the reassuring one.

Routing a walk has three segments:

| | |
| --- | --- |
| **D1** | the origin to the nearest network node |
| **D2** | the network distance between nodes |
| **D3** | the last node to the destination |

Fieldwork's isochrone **measures D1, records it in the receipt as `fw:networkSnaps`, and then
starts the snapped node at a cost of zero**. The walk onto the network is free, so the isochrone
reaches further than the stated time allows — by exactly the displacement this notebook prints.
D3 is not modelled at all; a uniform corridor buffer stands in for it.

On the shipped Soho isochrone the error is small, because 1854 Soho is densely noded: D1 is
10.9 m against a 375 m budget, about 2.9%.

**That number is the wrong one to remember.** D1 is largest exactly where a home sits back from
any mapped road — and that is where an access analysis is used to argue for investment. Ignoring
it flatters the places that are worst served.

**And past the snap limit the site is not understated — it is excluded.** `snap_to_network` and
`isochrone` both raise an error when the nearest node is further than `maxSnapM`, which ends the
whole run. Missing *road* data produces no answer rather than an `Unknown`, which inverts the rule
this project applies to every other kind of missing evidence. OSM footpath coverage is unequal in
a known direction — informal settlements are under-mapped — so the exclusion lands hardest on the
populations the analysis exists to make visible.

**The fallback, and what it may not claim.** Where a home is far from any road, the only available
route is a straight line. Fieldwork has that already: `travel_time` converts straight-line
distance at an assumed speed, and the access ruleset's predicate is named `fw:proxyMinutes` for
exactly this reason.

But a straight line assumes you can walk it — through yards, over walls, across a river or a
railway. Where D1 is large, the walked path is routinely longer than the crow flies. So a
crow-flies D1 is a **lower bound on the first segment, not a measurement of it**: charging it
makes the isochrone smaller and still optimistic. Charging it at street-walking speed is a second
optimistic assumption, since walking off-path is slower.

None of this is visible in the application, and deliberately so — it belongs here rather than
cluttering the Inspector with caveats nobody can act on mid-task. Fieldwork's design record is
[experiment 114](https://github.com/digital-twin-ph/fieldwork/blob/main/docs/experiments/114-the-three-segments-of-a-walk.md).


### Reading the result

`agrees` means both implementations chose the same junction for every pump and agree on how far
each moved, to the precision the widget publishes. It does not mean snapping is harmless, and it
does not mean the choice is robust — one of these eight was decided by 0.43 m.